<a href="https://colab.research.google.com/github/Ashik-13/PINNs/blob/main/Newton's_Cooling_Law_in_PINNs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Newton's Law of Cooling using Physics-Informed Neural Networks (PINNs)

## Overview

In this notebook, we solve **Newton's Law of Cooling** using a **Physics-Informed Neural Network (PINN)**.

Physics-Informed Neural Networks combine neural networks with the governing laws of physics. Instead of learning only from labeled data, a PINN incorporates the governing differential equation and the associated initial or boundary conditions directly into the loss function.

For this problem, the neural network will learn the temperature evolution of a cooling object while satisfying Newton's Law of Cooling.



## Objectives

The main objectives of this notebook are to:

- formulate Newton's Law of Cooling as an Ordinary Differential Equation (ODE),
- approximate the temperature field using a neural network,
- compute the required derivative using automatic differentiation,
- construct the physics residual,
- define physics and initial-condition losses,
- train the PINN using gradient-based optimization,
- compare the PINN prediction with the analytical solution,
- evaluate the prediction error,
- examine how well the trained model satisfies the governing physics.



## PINN Workflow

The overall workflow is

$$
t
\rightarrow
T_\theta(t)
\rightarrow
\frac{dT_\theta}{dt}
\rightarrow
\text{Physics Residual}
\rightarrow
\text{Loss Function}
\rightarrow
\text{Optimization}
$$

where

$$
T_\theta(t)
$$

is the temperature predicted by the neural network.



## Important Idea

The analytical solution will **not** be used to train the PINN.

The PINN will learn primarily from:

1. the governing differential equation, and
2. the initial condition.

The analytical solution will only be used after training to evaluate the accuracy of the PINN.

# 1. Physical Problem and Governing Equation

Newton's Law of Cooling states that the rate of change of the temperature of an object is proportional to the temperature difference between the object and its surroundings.

The governing equation is

$$
\frac{dT}{dt}
=
-k(T-T_{\text{ambient}})
$$

or equivalently,

$$
\boxed{
\frac{dT}{dt}
-
k(T_{\text{ambient}}-T)
=
0
}
$$

where:

- $T(t)$ = temperature of the object at time $t$
- $T_{\text{ambient}}$ = ambient or surrounding temperature
- $k$ = cooling constant
- $t$ = time
- $\frac{dT}{dt}$ = rate of change of temperature

Since the problem contains only one independent variable, $t$, this governing equation is an **Ordinary Differential Equation (ODE)**.



## Physical Parameters

For the present problem,

$$
T_{\text{ambient}} = 27^\circ C
$$

$$
T(0)=T_0=250^\circ C
$$

$$
k=0.45\;s^{-1}
$$

and the computational domain is

$$
0\leq t\leq10\;s.
$$

Therefore, the specific governing equation becomes

$$
\boxed{
\frac{dT}{dt}
-
0.45(27-T)
=
0
}
$$

subject to the initial condition

$$
\boxed{
T(0)=250^\circ C
}
$$



## Physical Interpretation

Initially,

$$
T_0 > T_{\text{ambient}}.
$$

Therefore, the object loses heat to its surroundings.

At the beginning, the temperature difference is large, so the cooling rate is relatively high.

As the temperature of the object approaches the ambient temperature, the temperature difference decreases and the cooling rate becomes smaller.

For sufficiently large time,

$$
T(t)\rightarrow T_{\text{ambient}}.
$$

Thus,

$$
\boxed{
T(t)\rightarrow27^\circ C
}
$$

as

$$
t\rightarrow\infty.
$$

# Import Required Libraries

In [ ]:
import numpy as np
import tensorflow as tf
import plotly.graph_objects as go

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


# Define Physical Parameters

In [ ]:
# Ambient temperature (°C)
T_amb = 27.0

# Initial temperature of the object (°C)
T_0 = 250.0

# Cooling constant (s^-1)
k = 0.45

# Computational time domain (s)
t_min = 0.0
t_max = 10.0

print("\nPhysical parameters:")
print(f"Ambient temperature : {T_amb} °C")
print(f"Initial temperature : {T_0} °C")
print(f"Cooling constant    : {k} s^-1")
print(f"Time domain         : {t_min} to {t_max} s")


Physical parameters:
Ambient temperature : 27.0 °C
Initial temperature : 250.0 °C
Cooling constant    : 0.45 s^-1
Time domain         : 0.0 to 10.0 s


# 2. Analytical Solution

The governing equation is

$$
\frac{dT}{dt}
=
-k(T-T_{\text{ambient}}).
$$

Separating the variables gives

$$
\frac{dT}{T-T_{\text{ambient}}}
=
-k\,dt.
$$

After integration,

$$
\ln|T-T_{\text{ambient}}|
=
-kt+C.
$$

Therefore,

$$
T-T_{\text{ambient}}
=
Ce^{-kt}.
$$

Using the initial condition

$$
T(0)=T_0,
$$

we obtain

$$
C=T_0-T_{\text{ambient}}.
$$

Hence, the analytical solution is

$$
\boxed{
T(t)
=
T_{\text{ambient}}
+
(T_0-T_{\text{ambient}})e^{-kt}
}
$$

For the present parameters,

$$
T_{\text{ambient}}=27^\circ C,
$$

$$
T_0=250^\circ C,
$$

and

$$
k=0.45\;s^{-1},
$$

so

$$
\boxed{
T(t)=27+223e^{-0.45t}
}
$$

This analytical solution will be used only as a reference for validating the trained PINN.

# Analytical Solution

In [ ]:
def analytical_solution(t):

    return T_amb + (T_0 - T_amb) * np.exp(-k * t)

# Generate points for visualization
t_analytical = np.linspace(t_min, t_max, 500)

T_analytical = analytical_solution(t_analytical)

# Interactive Analytical Solution Plot

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=t_analytical,
        y=T_analytical,
        mode="lines",
        name="Analytical Solution",
        hovertemplate=(
            "Time = %{x:.3f} s<br>"
            "Temperature = %{y:.3f} °C"
            "<extra></extra>"
        )
    )
)

# Ambient temperature
fig.add_hline(y=T_amb, line_dash="dash", annotation_text="Ambient Temperature")

fig.update_layout(
    title="Analytical Solution of Newton's Law of Cooling",
    xaxis_title="Time (s)",
    yaxis_title="Temperature (°C)",
    hovermode="x unified",
    template="plotly_white"
)

fig.show()

# PINN Formulation

In the PINN approach, the unknown temperature function is approximated by a neural network.

Instead of directly solving for

$$
T(t),
$$

we define

$$
\boxed{
T(t)\approx T_\theta(t)
}
$$

where $\theta$ represents all trainable parameters of the neural network, including its weights and biases.

The neural network therefore learns the mapping

$$
\boxed{
t
\rightarrow
T_\theta(t)
}
$$

where:

- input = time $t$
- output = predicted temperature $T_\theta(t)$


## Neural Network Architecture

A fully connected feed-forward neural network is used with the architecture

$$
1
\rightarrow
50
\rightarrow
50
\rightarrow
50
\rightarrow
1.
$$

This corresponds to:

- 1 input variable: $t$
- 3 hidden layers
- 50 neurons in each hidden layer
- 1 output variable: $T_\theta(t)$

The hidden layers use the **hyperbolic tangent (`tanh`) activation function**.


## Why use `tanh`?

PINNs require derivatives of the neural-network output.

For example, the governing equation requires

$$
\frac{dT_\theta}{dt}.
$$

Therefore, smooth and differentiable activation functions are particularly useful.

The `tanh` function is smooth and differentiable throughout its domain, which makes it a common activation function for PINNs.

# Create The Pinn Neural Network

In [ ]:
def create_model():

    model = tf.keras.Sequential([

        tf.keras.Input(shape=(1,)),

        tf.keras.layers.Dense(50, activation="tanh"),

        tf.keras.layers.Dense(50, activation="tanh"),

        tf.keras.layers.Dense(50, activation="tanh"),

        tf.keras.layers.Dense(1)
    ])

    return model

# Create the PINN
model = create_model()

# Display the neural-network architecture
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 50)             │           100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 50)             │         2,550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 50)             │         2,550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,251 (20.51 KB)

 Trainable params: 5,251 (20.51 KB)

 Non-trainable params: 0 (0.00 B)

# Automatic Differentiation and Physics Residual

The neural network predicts

$$
T_\theta(t),
$$

but Newton's Law of Cooling also requires the derivative

$$
\frac{dT_\theta}{dt}.
$$

Instead of calculating this derivative using a finite-difference approximation, TensorFlow uses **automatic differentiation**.

Thus,

$$
T_\theta(t)
\xrightarrow{\text{Automatic Differentiation}}\frac{dT_\theta}{dt}.
$$

## Physics Residual

The governing equation is

$$
\frac{dT}{dt} - k(T_{\text{ambient}}-T) = 0.
$$

Replacing the exact solution with the neural-network approximation gives

$$
\frac{dT_\theta}{dt} - k(T_{\text{ambient}}-T_\theta) = 0.
$$

Therefore, the physics residual is defined as

$$
\boxed{
f_\theta(t) = \frac{dT_\theta}{dt} - k(T_{\text{ambient}}-T_\theta)}
$$

If the neural-network prediction perfectly satisfies Newton's Law of Cooling,

$$
\boxed{
f_\theta(t)=0.
}
$$

During training, the PINN attempts to make this residual as close to zero as possible throughout the computational domain.

# Define The Physics Residual

In [ ]:
def physics_residual(model, t):

    # Record the operations involving time t
    with tf.GradientTape() as tape:

        # Tell TensorFlow to differentiate with respect to t
        tape.watch(t)

        # PINN temperature prediction
        T_pred = model(t)

    # Automatic differentiation:
    # calculate dT_pred/dt
    dT_dt = tape.gradient(T_pred, t)

    # Physics residual
    residual = dT_dt - k * (T_amb - T_pred)

    return residual

# 5. Collocation Points

A PINN needs locations inside the computational domain where the governing equation will be enforced.

These locations are known as **collocation points**.

For this problem, a set of time points is selected:

$$
t_1,t_2,\ldots,t_{N_f}.
$$

At every collocation point, the physics residual

$$
f_\theta(t_i)
$$

is calculated.

The objective of the PINN is to make

$$
f_\theta(t_i)\approx0
$$

for all collocation points.

In this notebook, we use

$$
N_f=100
$$

collocation points distributed uniformly between

$$
0\leq t\leq10\;s.
$$

An important distinction is that these points do **not** contain temperature labels.

The PINN receives the time coordinates and learns the temperature solution from the governing physics and the initial condition.

# Generate Collocation Points

In [ ]:
N_f = 100

t_train = np.linspace(t_min, t_max, N_f).reshape(-1, 1).astype(np.float32)

t_train = tf.convert_to_tensor(t_train, dtype=tf.float32)

# Define Initial-Condition Data

In [ ]:
# Initial time: t = 0
t_ic = tf.constant([[0.0]], dtype=tf.float32)

# Initial temperature: T(0) = 250 °C
T_ic = tf.constant([[T_0]], dtype=tf.float32)


print("Number of collocation points :", t_train.shape[0])
print("Initial time                 :", t_ic.numpy().flatten()[0], "s")
print("Initial temperature          :", T_ic.numpy().flatten()[0], "°C")

Number of collocation points : 100
Initial time                 : 0.0 s
Initial temperature          : 250.0 °C


# 6. PINN Loss Function

The loss function is the central mechanism through which the neural network learns the physical solution.

For Newton's Law of Cooling, two main constraints must be satisfied:

1. the governing differential equation,
2. the initial condition.

Therefore, two loss components are introduced.

## Physics Loss

The physics residual is

$$
f_\theta(t)
=
\frac{dT_\theta}{dt}
-
k(T_{\text{ambient}}-T_\theta).
$$

The physics loss is defined as the mean squared residual:

$$
\boxed{
\mathcal{L}_{physics}
=
\frac{1}{N_f}
\sum_{i=1}^{N_f}
f_\theta(t_i)^2
}
$$

or

$$
\mathcal{L}_{physics}
=
\text{MSE}(f_\theta).
$$

Minimizing this term forces the neural network to satisfy Newton's Law of Cooling.

## Initial-Condition Loss

The initial condition is

$$
T(0)=T_0.
$$

The neural network predicts

$$
T_\theta(0).
$$

Therefore,

$$
\boxed{
\mathcal{L}_{IC}
=
\left[T_\theta(0)-T_0\right]^2
}
$$

For this problem,

$$
\mathcal{L}_{IC}
=
\left[T_\theta(0)-250\right]^2.
$$

## Total Loss

The two components are combined as

$$
\boxed{
\mathcal{L}_{total}
=
w_{physics}\mathcal{L}_{physics}
+
w_{IC}\mathcal{L}_{IC}
}
$$

where $w_{physics}$ and $w_{IC}$ control the relative importance of the two constraints.

For the initial experiment,

$$
w_{physics}=1,
$$

$$
w_{IC}=1.
$$

Later, these weights can be varied to study the effect of loss balancing on PINN performance.

# Define Pinn Loss Function

In [ ]:
# Loss weights
w_physics = 1.0
w_ic = 1.0

def compute_losses(model, t_train, t_ic, T_ic):

    # Physics Loss

    residual = physics_residual(model, t_train)

    physics_loss = tf.reduce_mean(tf.square(residual))

    # Initial-Condition Loss

    T_ic_pred = model(t_ic)

    ic_loss = tf.reduce_mean(tf.square(T_ic_pred - T_ic))

    # Total Loss

    total_loss = (w_physics * physics_loss + w_ic * ic_loss)

    return total_loss, physics_loss, ic_loss

# PINN Training Strategy

Training the PINN means finding the neural-network parameters $\theta$ that minimize the total loss:

$$
\boxed{
\theta^*
=
\arg\min_\theta
\mathcal{L}_{total}
}
$$

During each training iteration:

1. time coordinates are passed through the neural network,
2. the network predicts $T_\theta(t)$,
3. automatic differentiation calculates $\frac{dT_\theta}{dt}$,
4. the physics residual is evaluated,
5. the physics loss is calculated,
6. the initial-condition loss is calculated,
7. the total loss is formed,
8. gradients of the loss are calculated,
9. the neural-network weights and biases are updated.

## Adam Optimizer

The **Adam optimizer** is used to update the neural-network parameters.

The initial learning rate is

$$
10^{-3}.
$$

An exponential learning-rate decay is also used.

At the beginning of training, relatively larger optimization steps help the network move quickly toward a suitable solution.

As training progresses, the learning rate gradually decreases, allowing finer adjustments to the neural-network parameters.

## Training Epochs

The network will initially be trained for

$$
15,000
$$

epochs.

During training, three quantities will be recorded:

- total loss,
- physics loss,
- initial-condition loss.

These will later be visualized using an interactive logarithmic loss plot.

#  Define Learning-Rate and Adam Optimizer

In [ ]:

#  DEFINE LEARNING-RATE SCHEDULE

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate=1e-3, decay_steps=500, decay_rate=0.95, staircase=False)

# 10. DEFINE ADAM OPTIMIZER

optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

# Define One Pinn Training Step

In [ ]:
@tf.function
def train_step():

    # Record operations required for backpropagation
    with tf.GradientTape() as tape:

        total_loss, physics_loss, ic_loss = compute_losses(model, t_train, t_ic, T_ic)

    # Calculate gradients of total loss with respect
    # to all trainable neural-network parameters
    gradients = tape.gradient(total_loss, model.trainable_variables)

    # Update weights and biases
    optimizer.apply_gradients(
        zip(gradients, model.trainable_variables)
    )

    return total_loss, physics_loss, ic_loss

## What Happens Inside One Training Step?

The complete sequence of operations is

$$
t
$$

$$
\downarrow
$$

$$
T_\theta(t)
$$

$$
\downarrow
$$

Automatic differentiation:

$$
\frac{dT_\theta}{dt}
$$

$$
\downarrow
$$

Physics residual:

$$
f_\theta(t)
=
\frac{dT_\theta}{dt}
-
k(T_{\text{ambient}}-T_\theta)
$$

$$
\downarrow
$$

Physics loss:

$$
\mathcal{L}_{physics}
$$

At the same time, the network evaluates

$$
T_\theta(0)
$$

and compares it with

$$
T_0=250^\circ C
$$

to calculate

$$
\mathcal{L}_{IC}.
$$

The total loss is then

$$
\mathcal{L}_{total}
=
\mathcal{L}_{physics}
+
\mathcal{L}_{IC}.
$$

Backpropagation calculates

$$
\frac{\partial\mathcal{L}_{total}}
{\partial\theta},
$$

and the optimizer updates the neural-network parameters.

This process is repeated thousands of times until the network approximately satisfies both the physics and the initial condition.

# Train The Pinn

In [ ]:
epochs = 15000

# Lists for storing training history
epoch_history = []
total_loss_history = []
physics_loss_history = []
ic_loss_history = []

for epoch in range(1, epochs + 1):

    total_loss, physics_loss, ic_loss = train_step()

    # Store losses every 50 epochs
    if epoch == 1 or epoch % 50 == 0:

        epoch_history.append(epoch)

        total_loss_history.append(
            float(total_loss.numpy())
        )

        physics_loss_history.append(
            float(physics_loss.numpy())
        )

        ic_loss_history.append(
            float(ic_loss.numpy())
        )


    # Display training progress every 1000 epochs
    if epoch == 1 or epoch % 1000 == 0:

        print(
            f"Epoch {epoch:5d} | "
            f"Total Loss = {total_loss.numpy():.6e} | "
            f"Physics Loss = {physics_loss.numpy():.6e} | "
            f"IC Loss = {ic_loss.numpy():.6e}"
        )

Epoch     1 | Total Loss = 6.264919e+04 | Physics Loss = 1.491881e+02 | IC Loss = 6.250000e+04
Epoch  1000 | Total Loss = 3.672152e+04 | Physics Loss = 2.062388e+02 | IC Loss = 3.651528e+04
Epoch  2000 | Total Loss = 2.453864e+04 | Physics Loss = 9.783325e+02 | IC Loss = 2.356031e+04
Epoch  3000 | Total Loss = 1.709772e+04 | Physics Loss = 2.036150e+03 | IC Loss = 1.506158e+04
Epoch  4000 | Total Loss = 1.267976e+04 | Physics Loss = 3.195068e+03 | IC Loss = 9.484696e+03
Epoch  5000 | Total Loss = 1.022730e+04 | Physics Loss = 4.327881e+03 | IC Loss = 5.899419e+03
Epoch  6000 | Total Loss = 9.019713e+03 | Physics Loss = 5.332817e+03 | IC Loss = 3.686896e+03
Epoch  7000 | Total Loss = 8.534930e+03 | Physics Loss = 6.123169e+03 | IC Loss = 2.411761e+03
Epoch  8000 | Total Loss = 8.397455e+03 | Physics Loss = 6.638573e+03 | IC Loss = 1.758882e+03
Epoch  9000 | Total Loss = 1.654848e+03 | Physics Loss = 3.303246e+02 | IC Loss = 1.324523e+03
Epoch 10000 | Total Loss = 2.779299e+02 | Physics 

# 8. Training-Loss Analysis

A successful training process should generally show a reduction in

$$
\mathcal{L}_{total},
$$

$$
\mathcal{L}_{physics},
$$

and

$$
\mathcal{L}_{IC}.
$$

The physics loss indicates how accurately the neural network satisfies the governing differential equation.

The initial-condition loss indicates how accurately the network satisfies

$$
T(0)=250^\circ C.
$$

Because PINN losses may span several orders of magnitude, a logarithmic vertical axis is useful for visualizing the convergence behavior.

# Interactive Training-Loss Plot

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(x=epoch_history, y=total_loss_history, mode="lines", name="Total Loss")
)

fig.add_trace(
    go.Scatter(x=epoch_history, y=physics_loss_history, mode="lines", name="Physics Loss")
)

fig.add_trace(
    go.Scatter(x=epoch_history, y=ic_loss_history, mode="lines", name="Initial Condition Loss")
)

fig.update_layout(title="PINN Training History", xaxis_title="Epoch", yaxis_title="Loss", yaxis_type="log", hovermode="x unified", template="plotly_white")

fig.show()

# PINN Prediction and Validation

After training, the neural network provides an approximation

$$
T_\theta(t)
$$

to the true temperature evolution.

To evaluate the model, predictions will be generated at 1000 points over the time interval

$$
0\leq t\leq10\;s.
$$

The PINN prediction will then be compared with the analytical solution

$$
T_{\text{exact}}(t)
=
T_{\text{ambient}}
+
(T_0-T_{\text{ambient}})e^{-kt}.
$$

For a successful PINN,

$$
\boxed{
T_\theta(t)\approx T_{\text{exact}}(t)
}
$$

throughout the computational domain.

A close overlap between the two curves provides visual evidence that the neural network has learned the correct physical solution.

In [ ]:
# GENERATE TEST POINTS

t_test = np.linspace(t_min, t_max, 1000).reshape(-1, 1).astype(np.float32)

t_test_tf = tf.convert_to_tensor(t_test, dtype=tf.float32)

# PINN PREDICTION

T_pinn = model(t_test_tf, training=False).numpy()

# ANALYTICAL SOLUTION

T_true = analytical_solution(t_test)

# Interactive Pinn vs Analytical Solution

In [ ]:
fig = go.Figure()

# Analytical solution
fig.add_trace(
    go.Scatter(
        x=t_test.flatten(),
        y=T_true.flatten(),
        mode="lines",
        name="Analytical Solution",
        hovertemplate=(
            "Time = %{x:.3f} s<br>"
            "T Analytical = %{y:.4f} °C"
            "<extra></extra>"
        )
    )
)

# PINN solution
fig.add_trace(
    go.Scatter(
        x=t_test.flatten(),
        y=T_pinn.flatten(),
        mode="lines",
        name="PINN Solution",
        line=dict(dash="dash"),
        hovertemplate=(
            "Time = %{x:.3f} s<br>"
            "T PINN = %{y:.4f} °C"
            "<extra></extra>"
        )
    )
)

# Ambient temperature reference
fig.add_hline(
    y=T_amb,
    line_dash="dot",
    annotation_text="Ambient Temperature"
)

fig.update_layout(
    title="Newton's Law of Cooling: PINN vs Analytical Solution",
    xaxis_title="Time (s)",
    yaxis_title="Temperature (°C)",
    hovermode="x unified",
    template="plotly_white"
)

fig.show()

# Quantitative Error Analysis

Visual agreement between the analytical and PINN solutions is useful, but numerical error metrics provide a more objective assessment.

The absolute error is defined as

$$
\boxed{
E(t)
=
|T_\theta(t)-T_{\text{exact}}(t)|
}
$$

Three error metrics will be evaluated.

### Mean Absolute Error

$$
\boxed{
MAE
=
\frac{1}{N}
\sum_{i=1}^{N}
|T_{\theta,i}-T_{\text{exact},i}|
}
$$

### Root Mean Squared Error

$$
\boxed{
RMSE
=
\sqrt{
\frac{1}{N}
\sum_{i=1}^{N}
(T_{\theta,i}-T_{\text{exact},i})^2
}
}
$$

### Maximum Absolute Error

$$
\boxed{
E_{\max}
=
\max
|T_\theta-T_{\text{exact}}|
}
$$

Smaller values of these metrics indicate better agreement between the PINN and the analytical solution.

# Calculate Prediction Error

In [ ]:
absolute_error = np.abs(T_pinn - T_true)

mae = np.mean(absolute_error)

rmse = np.sqrt(np.mean((T_pinn - T_true) ** 2))

max_error = np.max(absolute_error)

print("PINN Error Metrics")
print("-----------------------------")
print(f"Mean Absolute Error : {mae:.6f} °C")
print(f"RMSE                : {rmse:.6f} °C")
print(f"Maximum Error       : {max_error:.6f} °C")

PINN Error Metrics
-----------------------------
Mean Absolute Error : 0.452100 °C
RMSE                : 0.670818 °C
Maximum Error       : 1.854111 °C


# Interactive Absolute Error Plot

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=t_test.flatten(),
        y=absolute_error.flatten(),
        mode="lines",
        name="Absolute Error",
        hovertemplate=(
            "Time = %{x:.3f} s<br>"
            "Absolute Error = %{y:.6f} °C"
            "<extra></extra>"
        )
    )
)

fig.update_layout(
    title="Absolute Error: PINN vs Analytical Solution",
    xaxis_title="Time (s)",
    yaxis_title="Absolute Error (°C)",
    template="plotly_white"
)

fig.show()

# 11. Physics Residual Validation

Matching the analytical solution is not the only requirement for a Physics-Informed Neural Network.

The trained model should also satisfy the original governing differential equation.

The residual is

$$
f_\theta(t)
=
\frac{dT_\theta}{dt}
-
k(T_{\text{ambient}}-T_\theta).
$$

For an ideal solution,

$$
f_\theta(t)=0.
$$

Therefore, after training we evaluate the residual throughout the entire domain.

A well-trained PINN should satisfy

$$
\boxed{
|f_\theta(t)|\approx0
}
$$

for most of the computational domain.

This provides a direct measure of how well the learned solution respects the underlying physics.

# Calculate Physics Residual

In [ ]:
residual_test = physics_residual(model, t_test_tf).numpy()

# Interactive Physics Residual Plot

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=t_test.flatten(),
        y=residual_test.flatten(),
        mode="lines",
        name="Physics Residual",
        hovertemplate=(
            "Time = %{x:.3f} s<br>"
            "Residual = %{y:.6e}"
            "<extra></extra>"
        )
    )
)

fig.add_hline(y=0, line_dash="dash",annotation_text="Ideal Residual = 0")

fig.update_layout(
    title="Physics Residual of the Trained PINN",
    xaxis_title="Time (s)",
    yaxis_title="ODE Residual",
    template="plotly_white"
)

fig.show()

# Initial-Condition Verification

The PINN was trained with the constraint

$$
T(0)=250^\circ C.
$$

Therefore, the final trained model should predict

$$
T_\theta(0)\approx250^\circ C.
$$

The difference

$$
\boxed{
E_{IC}
=
|T_\theta(0)-T_0|
}
$$

provides a direct measure of how accurately the initial condition has been satisfied.

# Verify The Initial Condition

In [ ]:
T0_predicted = model(
    tf.constant([[0.0]], dtype=tf.float32), training=False).numpy()[0, 0]

initial_condition_error = abs(T0_predicted - T_0)

print("Initial-Condition Verification")
print("--------------------------------------")
print(f"Required T(0)       : {T_0:.6f} °C")
print(f"PINN predicted T(0) : {T0_predicted:.6f} °C")
print(f"Absolute IC error   : {initial_condition_error:.6f} °C")

Initial-Condition Verification
--------------------------------------
Required T(0)       : 250.000000 °C
PINN predicted T(0) : 249.998016 °C
Absolute IC error   : 0.001984 °C


# Discussion and Conclusion

In this notebook, Newton's Law of Cooling was solved using a Physics-Informed Neural Network.

The governing ODE was

$$
\frac{dT}{dt}
-
k(T_{\text{ambient}}-T)
=
0
$$

with the initial condition

$$
T(0)=250^\circ C.
$$

The neural network approximated the unknown temperature function as

$$
T(t)\approx T_\theta(t).
$$

Automatic differentiation was used to calculate

$$
\frac{dT_\theta}{dt},
$$

allowing the governing differential equation to be incorporated directly into the loss function.

The total PINN loss consisted of two components:

$$
\mathcal{L}_{total}
=
\mathcal{L}_{physics}
+
\mathcal{L}_{IC}.
$$

The trained PINN was then evaluated using:

- comparison with the analytical solution,
- Mean Absolute Error,
- Root Mean Squared Error,
- Maximum Absolute Error,
- physics residual,
- initial-condition error.

## Key Learning Outcomes

This example demonstrates several fundamental ideas of Physics-Informed Neural Networks:

1. A neural network can represent the unknown solution of a differential equation.

2. Automatic differentiation allows derivatives of the neural-network output to be calculated directly.

3. A governing differential equation can be converted into a physics-based loss function.

4. Initial or boundary conditions can be incorporated as additional loss terms.

5. A PINN can learn a physical solution without requiring a conventional labeled temperature dataset throughout the domain.

6. The quality of a PINN should be evaluated using both prediction accuracy and physics consistency.

## Final PINN Concept

The essential idea of the method can be summarized as

$$
\boxed{
\text{Neural Network}
+
\text{Governing Physics}
+
\text{Initial/Boundary Conditions}
=
\text{Physics-Informed Neural Network}
}
$$

For this problem,

$$
\boxed{
t
\rightarrow
T_\theta(t)
\rightarrow
\frac{dT_\theta}{dt}
\rightarrow
\text{Physics Residual}
\rightarrow
\text{Loss Minimization}
}
$$

and after successful training,

$$
\boxed{
T_\theta(t)
\approx
T_{\text{analytical}}(t)
}
$$

while simultaneously satisfying

$$
\boxed{
f_\theta(t)\approx0.
}
$$